# OEC MusicGen — Gradio client for the isolated model server

**UI kernel:** \`phase1\` (Gradio + requests only for the notebook).  
**Inference backend:** the managed \`audiocraft-api\` sidecar on \`127.0.0.1:7875\`.

Start it from the OEC repository root on an admitted idle A100:
\`\`\`bash
python tools/oec_service_manager.py start audiocraft-api --provider musicgen --gpu <ID> --port 7875 --allow-download --accept-noncommercial-license
python tools/oec_service_manager.py status audiocraft-api
\`\`\`
Use \`--allow-download\` only if a cache fill is needed and fresh StorageManager admission allows it. AudioCraft weights have a CC-BY-NC 4.0 non-commercial use boundary. The UI itself loads no model weights and never installs Python packages.

## What this UI verifies

- Requests \`/health\`, requires \`ready=true\` and \`model_loaded=true\`, submits \`text_to_music\` to \`/v1/jobs\` and polls the real worker.
- Downloads the finished WAV, decodes it, checks sample rate/channels/frame count, and reports the SHA-256.
- Uses the isolated AudioCraft backend, instead of spawning a second model subprocess from the notebook.
- Runs one job at a time, binds only to loopback, and keeps result scratch files under \`/tmp/oec-notebook-results\`.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import time
import urllib.parse
import wave

import gradio as gr
import requests

API_BASE = os.environ.get("OEC_MEDIA_API_URL", "http://127.0.0.1:7875").rstrip("/")
RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if urllib.parse.urlsplit(API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("MusicGen notebook only permits a loopback model-server endpoint.")
HTTP = requests.Session()
HTTP.trust_env = False

def service_health():
    try:
        response = HTTP.get(f"{API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(f"MusicGen server is unreachable at {API_BASE}; start audiocraft-api first.") from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"MusicGen server is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload

def submit_and_wait(payload, timeout_seconds=900):
    health = service_health()
    try:
        response = HTTP.post(f"{API_BASE}/v1/jobs", json=payload, timeout=30)
    except Exception as exc:
        raise gr.Error(f"MusicGen job submission failed: {type(exc).__name__}") from exc
    if response.status_code != 202:
        raise gr.Error(f"MusicGen job rejected (HTTP {response.status_code}): {response.text[:1000]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected MusicGen job response: {accepted}")
    deadline = time.monotonic() + timeout_seconds
    latest = accepted
    while time.monotonic() < deadline:
        poll = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"MusicGen status lookup failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"MusicGen job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("MusicGen generation timed out; inspect audiocraft-api service logs.")
    result = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}/result", timeout=120)
    if result.status_code != 200:
        raise gr.Error(f"MusicGen result download failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "").split(";", 1)[0].lower()
    if content_type != "audio/wav":
        raise gr.Error(f"Expected audio/wav from MusicGen, received {content_type!r}.")
    output_path = RESULT_ROOT / f"{job_id}.wav"
    if output_path.exists():
        raise gr.Error("Refusing to overwrite an existing notebook result.")
    output_path.write_bytes(result.content)
    try:
        with wave.open(str(output_path), "rb") as audio:
            frames, rate, channels = audio.getnframes(), audio.getframerate(), audio.getnchannels()
    except Exception as exc:
        output_path.unlink(missing_ok=True)
        raise gr.Error("MusicGen returned a WAV file that cannot be decoded.") from exc
    if output_path.stat().st_size < 64 or frames <= 0 or rate <= 0 or channels <= 0:
        output_path.unlink(missing_ok=True)
        raise gr.Error("MusicGen returned an empty or invalid WAV.")
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": API_BASE, "health": health,
        "metadata": latest.get("metadata", {}), "artifact_path": str(output_path),
        "artifact_bytes": output_path.stat().st_size,
        "artifact_sha256": hashlib.sha256(output_path.read_bytes()).hexdigest(),
        "sample_rate_hz": rate, "channels": channels, "duration_seconds": frames / rate,
        "licence_boundary": "CC-BY-NC-4.0; non-commercial use unless independently confirmed",
    }
    return str(output_path), json.dumps(details, indent=2, ensure_ascii=False)

def generate_music(prompt, duration, seed):
    if not isinstance(prompt, str) or not prompt.strip():
        raise gr.Error("Enter a music prompt.")
    if not 1 <= int(duration) <= 10:
        raise gr.Error("Duration must be between 1 and 10 seconds.")
    seed = int(seed)
    if not 0 <= seed <= 2**31 - 1:
        raise gr.Error("Seed must be between 0 and 2147483647.")
    return submit_and_wait({
        "task": "text_to_music", "prompt": prompt.strip(),
        "parameters": {"duration_seconds": int(duration), "seed": seed},
    })

demo = gr.Interface(
    fn=generate_music,
    inputs=[
        gr.Textbox(label="Music prompt", lines=3, value="Calm instrumental soundtrack with warm piano and soft strings"),
        gr.Slider(1, 10, value=3, step=1, label="Duration (seconds)"),
        gr.Number(value=42, precision=0, label="Seed"),
    ],
    outputs=[
        gr.Audio(type="filepath", label="Generated WAV"),
        gr.Textbox(label="Run metadata", lines=12),
    ],
    title="OEC MusicGen — model server",
    description="Thin Gradio client; AudioCraft inference runs in the isolated backend.",
)
if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo.queue(max_size=1).launch(server_name="127.0.0.1", server_port=7902, share=False, inline=False, show_error=True)